Tensorflow and Keras were installed following the official tutorial:
https://www.tensorflow.org/install/pip

In [ ]:
import tensorflow as tf
from tensorflow import keras

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import sys
sys.path.append("..")
import methods.explainability as ex
from methods import utils

physical_devices = tf.config.list_physical_devices("GPU")
print("Num GPUs:", len(physical_devices))

In [ ]:
NOTEBOOK = "2D_keras_binary_L"
NORM = "none"
RES = 0.05
plane = "XY"
DIMS = "2D"
k_cv = 5
IMGS_DIR = f"../../data/preprocessed/2D_res={RES}_norm={NORM}_{k_cv}fold_withDMSO"

CNN_DENSE_FILTS = (256, 64, 16)
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 64
CHANNEL_MODE = "grayscale"
N_OUTPUT_UNITS = 1
LR_SCHED = None
OPTIMIZER = "AdamW"
COMMENT = "testing baseline"

EPOCHS = 50
LR = 1e-4
SEED = 2023

if CHANNEL_MODE == "rgb":
    channels = (3,)
elif CHANNEL_MODE == "grayscale":
    channels = (1,)

if N_OUTPUT_UNITS == 1:
    OUTPUT_FUNC = "sigmoid"
    LOSS_FUNC = tf.keras.losses.BinaryCrossentropy(
        label_smoothing=0.1,
    )
    LABEL_MODE = "binary"
elif N_OUTPUT_UNITS == 2:
    OUTPUT_FUNC = "softmax"
    LOSS_FUNC = "categorical_crossentropy"
    LABEL_MODE = "categorical"

In [ ]:
sns.color_palette("colorblind")
pal = utils.get_class_palette()
mic_pal = utils.get_microscopist_palette()

In [ ]:
k = 1 
train_ds = None

for i in range(1, k_cv + 1):
    if i == k:
        val_ds = tf.keras.utils.image_dataset_from_directory(
            f"{IMGS_DIR}/fold_{i}/",
            color_mode=CHANNEL_MODE,
            labels="inferred",
            label_mode=LABEL_MODE,
            interpolation="bilinear",
            seed=SEED,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            shuffle=False,
        )
        continue

    if train_ds:
        train_ds = train_ds.concatenate(
            tf.keras.utils.image_dataset_from_directory(
                f"{IMGS_DIR}/fold_{i}/",
                color_mode=CHANNEL_MODE,
                labels="inferred",
                label_mode=LABEL_MODE,
                interpolation="bilinear",
                seed=SEED,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                shuffle=True,
            )
        )
    else:
        train_ds = tf.keras.utils.image_dataset_from_directory(
            f"{IMGS_DIR}/fold_{i}/",
            color_mode=CHANNEL_MODE,
            labels="inferred",
            label_mode=LABEL_MODE,
            interpolation="bilinear",
            seed=SEED,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            shuffle=True,
        )

n_train_ims = train_ds.cardinality().numpy() * BATCH_SIZE
print(n_train_ims)

train_ds = train_ds.unbatch().shuffle(10000).batch(BATCH_SIZE)
# Prefetching samples in GPU memory helps maximize GPU utilization.
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
model = keras.models.load_model("../models/chromagenet/calmodel_fold_1.keras")
model.load_weights("../models/checkpoints/fold_1/44-0.692.weights.h5")

In [ ]:
model.evaluate(val_ds)

In [ ]:
train_ids, train_ds = utils.dataset_from_partition_cv(
    IMGS_DIR, ["fold_2", "fold_3", "fold_4", "fold_5"], CHANNEL_MODE, 
    LABEL_MODE, IMAGE_SIZE, BATCH_SIZE, SEED
)
val_ids, val_ds = utils.dataset_from_partition(
    IMGS_DIR,
    "fold_1",
    CHANNEL_MODE,
    LABEL_MODE,
    IMAGE_SIZE,
    BATCH_SIZE,
    SEED,
)

In [ ]:
train_df = ex.get_output_df_voting(train_ids, train_ds, model, LABEL_MODE)
train_df["class"] = train_df["label"].map({0: 'aged', 1: 'young'})
val_df = ex.get_output_df_voting(val_ids, val_ds, model, LABEL_MODE)
val_df["class"] = val_df["label"].map({0: 'aged', 1: 'young'})

In [ ]:
czi_metadata_filt = pd.read_csv("../results/czi_metadata_filt.csv")

In [ ]:
czi_metadata_filt = pd.read_csv("../results/czi_metadata_filt.csv")
czi_metadata_filt = czi_metadata_filt[czi_metadata_filt["condition"].isin(["aged", "aged_DMSO", "young"])]
czi_metadata_filt

In [ ]:
czi_metadata_filt.columns

In [ ]:
import re

nuc_ids = []

for f in czi_metadata_filt["path"].to_list():

    # Find the number of the nucleus while fighting inconsistencies in the data nomenclature
    if re.search(r"([ -]\d+\.czi)", f):
        nuc_num = re.split(r"(\d+\.czi)", f)[1]
        nuc_num = nuc_num.split(".")[0]
    else:
        nuc_num = re.split(r" (\d+\_)", f)[1]
        nuc_num = nuc_num.split("_")[0]

    batch_id = f.split("/")[4]
    nuc_ids.append(batch_id + "_nuc_" + nuc_num)

czi_metadata_filt["nuc_id"] = nuc_ids

In [ ]:
fixed = []

for f in val_df["nuc_id"]:
    # Find the number of the nucleus while fighting inconsistencies in the data nomenclature
    if re.search(r"-\d+$", f):
        print(f)
        nuc_batch, nuc_num = re.split(r"(\d+$)", f)[:2]
        nuc_batch = nuc_batch[:-1]
        nuc_batch = "_".join(nuc_batch.split("_")[:-1])
        f = "_nuc_".join([nuc_batch, nuc_num])
        print(f)

    elif re.search(r"_nuc_\d+_", f):
        nuc_batch, nuc_num = re.split(r"(_nuc_\d+_)", f)[:2]
        nuc_num = nuc_num[:-1]
        f = "".join([nuc_batch, nuc_num])
        
    fixed.append(f)

In [ ]:
for i, f in enumerate(fixed):
    if f not in czi_metadata_filt["nuc_id"].to_list():
        print(f.split("_"))
        a, b, c, d, e = f.split("_")
        fixed[i] = "_".join([a, b, d, e])

val_df["nuc_id"] = fixed

In [ ]:
val_df = val_df.merge(czi_metadata_filt, on="nuc_id")

In [ ]:
val_df.columns

In [ ]:
utils.plot_regplot(val_df, x="mean_prob", y="pixel_mean", hue="acquired_by",
             y_label="Original mean pixel intensity", palette=mic_pal, legend=True)

In [ ]:
utils.plot_regplot(val_df, x="mean_prob", y="num_Zs", hue="acquired_by",
             y_label="Original number of Z stacks", palette=mic_pal)

In [ ]:
utils.plot_regplot(val_df, x="mean_prob", y="sigma_noise", hue="acquired_by",
             y_label="Original estimated noise", palette=mic_pal)

In [ ]:
utils.plot_regplot(val_df, x="mean_prob", y="pixel_sum", hue="acquired_by",
             y_label="Original total pixel intensity", palette=mic_pal)

In [ ]:
plt.figure(figsize=(5, 3))
sns.violinplot(
    data=val_df,
    y="mean_prob",
    x="num_channels",
    hue="condition",
    palette=pal,
)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel("Number of channels in original image")
plt.ylabel("p(young)")

In [ ]:
plt.figure(figsize=(5, 3))
sns.violinplot(
    data=val_df,
    y="mean_prob",
    x="acquired_by",
    hue="condition",
    palette=pal,
)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel("Number of channels in original image")
plt.ylabel("p(young)")

In [ ]:
plt.figure(figsize=(5, 3))
sns.boxplot(
    data=val_df,
    y="mean_prob",
    x="year",
    hue="condition",
    palette=pal,
)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel("Number of channels in original image")
plt.ylabel("p(young)")